In [1]:
from pathlib import Path
from collections import Counter
import random
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.auto import tqdm

sns.set(style="whitegrid")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

# Point directly to your actual dataset locations
TRASHNET_DIR = Path(
    r"C:\Users\Manojkumar Mohankuma\OneDrive\Desktop\InfinyAI\Computer Vision & Machine Learning for Automated Sorting of Precious-Metal Scrap\Dataset\TrashNet++ A Real-Time Multi-Class Waste Recogniti"
)

KAGGLE_DIR = Path(
    r"C:\Users\Manojkumar Mohankuma\OneDrive\Desktop\InfinyAI\Computer Vision & Machine Learning for Automated Sorting of Precious-Metal Scrap\Dataset\modified-dataset"
)

print("TrashNet path exists:", TRASHNET_DIR.exists())
print("Kaggle path exists:  ", KAGGLE_DIR.exists())

TrashNet path exists: True
Kaggle path exists:   True


c:\Users\Manojkumar Mohankuma\OneDrive\Desktop\InfinyAI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def count_images_by_folder(root_dir):
    rows = []
    root_dir = Path(root_dir)
    if not root_dir.exists():
        return pd.DataFrame(columns=["folder", "image_count"])

    for folder in sorted(root_dir.rglob("*")):
        if folder.is_dir():
            image_count = sum(
                file.suffix.lower() in IMAGE_EXTENSIONS
                for file in folder.iterdir()
                if file.is_file()
            )
            if image_count > 0:
                rows.append({
                    "folder": str(folder.relative_to(root_dir)),
                    "image_count": image_count
                })

    return pd.DataFrame(rows)

trashnet_counts = count_images_by_folder(TRASHNET_DIR)
kaggle_counts = count_images_by_folder(KAGGLE_DIR)

print("TrashNet image counts")
display(trashnet_counts)

print("\nKaggle dataset image counts")
display(kaggle_counts)

TrashNet image counts


""



Kaggle dataset image counts


,folder,image_count
0,test\Battery,30
1,test\Keyboard,30
2,test\Microwave,30
3,test\Mobile,30
4,test\Mouse,30
5,test\PCB,30
6,test\Player,30
7,test\Printer,30
8,test\Television,30
9,test\Washing Machine,30


In [3]:
CLASS_MAPPING = {
    "Battery": "battery",
    "Keyboard": "keyboard",
    "Microwave": "microwave",
    "Mobile": "mobile",
    "Mouse": "mouse",
    "PCB": "pcb",
    "Player": "player",
    "Printer": "printer",
    "Television": "television",
    "Washing Machine": "washing_machine",
}

def build_kaggle_dataframe(root_dir):
    root_dir = Path(root_dir)
    rows = []

    for split in ["train", "val", "test"]:
        split_dir = root_dir / split
        if not split_dir.exists():
            continue

        for class_dir in sorted(split_dir.iterdir()):
            if not class_dir.is_dir():
                continue

            orig_class = class_dir.name
            mapped_class = CLASS_MAPPING.get(orig_class, orig_class)

            for img_path in class_dir.rglob("*"):
                if img_path.is_file() and img_path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append({
                        "split": split,
                        "orig_class": orig_class,
                        "class": mapped_class,
                        "path": str(img_path),
                    })

    return pd.DataFrame(rows)

kaggle_df = build_kaggle_dataframe(KAGGLE_DIR)
kaggle_df.head()

,split,orig_class,class,path
0,train,Battery,battery,C:\Users\Manojkumar Mohankuma\OneDrive\Desktop...
1,train,Battery,battery,C:\Users\Manojkumar Mohankuma\OneDrive\Desktop...
2,train,Battery,battery,C:\Users\Manojkumar Mohankuma\OneDrive\Desktop...
3,train,Battery,battery,C:\Users\Manojkumar Mohankuma\OneDrive\Desktop...
4,train,Battery,battery,C:\Users\Manojkumar Mohankuma\OneDrive\Desktop...


In [4]:
print(kaggle_df["split"].value_counts())
print()
print(kaggle_df["class"].value_counts())

split
train    2400
val       300
test      300
Name: count, dtype: int64

class
battery            300
keyboard           300
microwave          300
mobile             300
mouse              300
pcb                300
player             300
printer            300
television         300
washing_machine    300
Name: count, dtype: int64
